Use wb_command to resample NSD fsaverage NCSNR to fsLR32k space

This notebook resamples NSD subject specific NCSNR estimates from fsaverage to fsLR32k space. It can be divided into three parts:
1. Visualize subject specific NCSNR on fsaverage flatmap
2. Resample fsaverage map to fsLR32k space
3. Visualize subject specific ROI classes on fsLR32k flatmap

In [ ]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())
import os
import sys
sys.path.append(os.getenv('PYTHONPATH'))
import subprocess
from pathlib import Path
import nibabel as nib
import numpy as np
import hcp_utils as hcp
import matplotlib.pyplot as plt
from nilearn import plotting, datasets
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
import cortex
cortex.download_subject('fsaverage')

In [ ]:
#housekeeping
templateflow_dir = os.path.join("/data","vision","oliva","blahner","templateflow")
subjects = [f"sub-{x:02d}" for x in range(1,9)]

Plot fsaverage NCSNR from NSD. These fsaverage NCSNR files are available on NSD's data release in: nsddata_betas/ppdata/subj${sub}/fsaverage/betas_fithrf_GLMdenoise_RR/{l/r}h.ncsnr.mgh

In [ ]:

save_plot = False
for subject in subjects:
    # Load the ROI labels
    lh_ncsnr = np.squeeze(nib.load(os.path.join(os.getenv("PROJECT_ROOT"), "src", "fmriDatasetPreparation", "datasets", "NaturalScenesDataset", "validation", "output", "ncsnr_original", subject,
        f"lh.ncsnr.mgh")).get_fdata())
    rh_ncsnr = np.squeeze(nib.load(os.path.join(os.getenv("PROJECT_ROOT"), "src", "fmriDatasetPreparation", "datasets", "NaturalScenesDataset", "validation", "output", "ncsnr_original", subject,
        f"rh.ncsnr.mgh")).get_fdata())

    data = np.append(lh_ncsnr, rh_ncsnr)
    print(f"subject max ncsnr: {np.nanmax(data)}")
    vertex_data = cortex.Vertex(data, 'fsaverage', cmap='hot', vmin=0, vmax=max(data))

    # Plot the figure without colorbar
    fig = cortex.quickshow(vertex_data,
        with_curvature=True,
        curvature_brightness=0.5,
        with_rois=False,
        with_labels=False,
        linewidth=5,
        linecolor=(1, 1, 1),
        with_colorbar=False
    )

    #fig.suptitle(f'{subject}', fontsize=16)

    # Add colorbar separately with better positioning
    # Get the main axis
    ax = fig.axes[0]

    # Create a new axis for the colorbar
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="3%", pad=0.5)

    # Add the colorbar
    cbar = plt.colorbar(ax.images[0], cax=cax)
    cbar.set_label('NCSNR', rotation=270, labelpad=20)

    plt.tight_layout()
    if save_plot:
        plt.savefig(os.path.join(os.getenv("PROJECT_ROOT"), "src", "fmriDatasetPreparation", "datasets", "NaturalScenesDataset", "validation", "output", "plots", f"{subject}_fsaverage_nsdOriginal.png"), dpi=300)
    plt.show()

Project the fsaverage ncsnr to fsLR32k space

In [ ]:
working_path = os.path.join(os.getenv("PROJECT_ROOT"), "src", "fmriDatasetPreparation", "datasets", "NaturalScenesDataset", "validation")
largest = True # the -largest flag in metric-resample docs. with barycentric method, this makes the resampling nearest neighbors
method = "BARYCENTRIC"
for subject in subjects:
    for hemi in ['l', 'r']:
        # File paths
        temp_metric_in = os.path.join(working_path, "output", "ncsnr_original", subject, f"{hemi}h.ncsnr_fsaverage_space.func.gii")
        temp_metric_out = os.path.join(working_path, "output", "ncsnr_original", subject, f"{hemi}h.ncsnr_fsLR32k_space_resampled.func.gii")
        final_npy_out = os.path.join(working_path, "output", "ncsnr_original", subject, f"{hemi}h.ncsnr_fsLR32k_space_resampled.npy")

        current_sphere = os.path.join(templateflow_dir, "tpl-fsaverage", f"tpl-fsaverage_hemi-{hemi.upper()}_den-164k_sphere.surf.gii")
        new_sphere = os.path.join(templateflow_dir, "tpl-fsLR", f"tpl-fsLR_space-fsaverage_hemi-{hemi.upper()}_den-32k_sphere.surf.gii")

        #step 1: load data and convert to GIFTI
        data = np.squeeze(nib.load(os.path.join(working_path, "output", "ncsnr_original", subject,
            f"{hemi}h.ncsnr.mgh")).get_fdata())
        print(f"Original data shape: {data.shape}")

        # Create GIFTI metric image
        gifti_img = nib.gifti.GiftiImage()
        darray = nib.gifti.GiftiDataArray(data.astype(np.float32), intent='NIFTI_INTENT_CORREL') #the default int64 gives issues
        gifti_img.add_gifti_data_array(darray)

        # Save temporary GIFTI file
        nib.save(gifti_img, temp_metric_in)

        # Step 2: Run wb_command metric resample
        if largest:
            cmd = f"wb_command -metric-resample {temp_metric_in} {current_sphere} {new_sphere} {method} {temp_metric_out} -largest"
        else:
            cmd = f"wb_command -metric-resample {temp_metric_in} {current_sphere} {new_sphere} {method} {temp_metric_out}"

        print(f"Running command: {cmd}")
        subprocess.run(cmd, shell=True, check=True)

        # Step 3: Load resampled GIFTI and convert back to numpy
        resampled_gifti = nib.load(temp_metric_out)
        resampled_data = resampled_gifti.darrays[0].data
        print(f"Resampled data shape: {resampled_data.shape}")

        # Save as NumPy array
        np.save(final_npy_out, resampled_data)

        #clean up temp files
        os.remove(temp_metric_in)
        os.remove(temp_metric_out)

        # checks
        print(f"Input min: {data.min()}, max: {data.max()}")
        print(f"Input unique values count: {len(np.unique(data))}")
        print(f"Resampled min: {resampled_data.min()}, max: {resampled_data.max()}")
        print(f"Resampled unique values count: {len(np.unique(resampled_data))}")

        # check if all resampled values exist in original
        all_present = np.all(np.isin(resampled_data, data))
        print(f"All resampled values present in original: {all_present}")
        if all_present:
            print(f"{subject} Successfully resampled from fsaverage ({data.shape[0]} vertices) to fsLR32k ({resampled_data.shape[0]} vertices)")

Now visualize the projected fsaverage rois in fsLR32k space

In [ ]:
def plot_flatmap(stat, filepath=None, cmap='hot'):
    cortex_data_left = hcp.left_cortex_data(stat)
    cortex_data_right = hcp.right_cortex_data(stat)

    #determine global min/max for consistent color scaling
    datamin = min(np.nanmin(cortex_data_left), np.nanmin(cortex_data_right))
    datamax = max(np.nanmax(cortex_data_left), np.nanmax(cortex_data_right))
    threshold = None
    vmin=datamin
    vmax=datamax
    #create a figure with multiple axes to plot each anatomical image
    fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(10, 4), subplot_kw={'projection': '3d'})
    plt.subplots_adjust(wspace=0)
    im = plotting.plot_surf(hcp.mesh.flat_left, cortex_data_left,
            threshold=threshold, bg_map=hcp.mesh.sulc_left, 
            colorbar=False, cmap=cmap, 
            vmin=vmin, vmax=vmax,
            axes = axes[0])
    im = plotting.plot_surf(hcp.mesh.flat_right, cortex_data_right,
            threshold=threshold, bg_map=hcp.mesh.sulc_right, 
            colorbar=False, cmap=cmap, 
            vmin=vmin, vmax=vmax,
            axes = axes[1])
    
    #flip along the horizontal
    axes[0].invert_yaxis()
    axes[1].invert_yaxis()

    #create colorbar
    norm = plt.Normalize(vmin=vmin, vmax=vmax)
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=axes.ravel().tolist(), shrink=0.6)

    cbar.set_ticks([0, round(datamax,2)])
    cbar.set_ticklabels([0, round(datamax,2)])
    if filepath:
        if Path(filepath).suffix == 'png':
             plt.savefig(filepath, dpi=340)
        else:
             plt.savefig(filepath)
    plt.show()

In [ ]:
for subject in subjects:
    lh_ncsnr_resampled = np.load(os.path.join(working_path, "output", "ncsnr_original", subject, f"lh.ncsnr_fsLR32k_space_resampled.npy"))
    rh_ncsnr_resampled = np.load(os.path.join(working_path, "output", "ncsnr_original", subject, f"rh.ncsnr_fsLR32k_space_resampled.npy"))

    #resampled left and right are shape (32491,) already mapped to the mesh. I'm essentially just doing the unmapping so my regular plotting functions will work with the hcp.get_left/right_cortex() functions to turn it back into a mesh needed for plotting
    vertex_info = hcp.vertex_info
    resampled_data = np.hstack((lh_ncsnr_resampled[vertex_info.grayl],rh_ncsnr_resampled[vertex_info.grayr]))
    plot_flatmap(resampled_data,
                filepath=os.path.join(os.getenv("PROJECT_ROOT"), "src", "fmriDatasetPreparation", "datasets", "NaturalScenesDataset", "validation", "output", "plots", "ncsnr_nsd_fsLR32k", f"{subject}_ncsnr_original_fsLR32k.png")
    )